# Paper #28 Implementation: Observations of Cool-Star Magnetic Fields / 저온 별 자기장 관측

Based on Reiners, A. (2012), *Living Reviews in Solar Physics*, DOI: 10.12942/lrsp-2012-1.

이 노트북은 저온 별 자기장 측정의 핵심 개념들을 구현한다:
This notebook implements the core concepts for measuring cool-star magnetic fields:

1. Zeeman 분리 계산 (Δλ_B = 4.67e-13 λ² g B) / Zeeman splitting
2. Stokes I, V 프로파일 합성 / Stokes I and V profile synthesis
3. 광학 vs 근적외선 민감도 비교 / Optical vs near-IR sensitivity
4. Zeeman vs 회전 도플러 확대 임계값 / Zeeman vs rotational broadening threshold
5. Rossby 수와 활동도 관계 / Rossby number and activity relation
6. Zeeman Doppler Imaging 개념 / Zeeman Doppler Imaging concept

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

C_KMS = 2.998e5

## 1. Zeeman Splitting Calculator / Zeeman 분리 계산

정상 Zeeman 효과에 의한 파장 분리:
Wavelength splitting from normal Zeeman effect:

$$
\Delta\lambda_B = \frac{e}{4\pi m_e c} g \lambda^2 B = 4.67 \times 10^{-13}\, g\, \lambda^2\, B
$$

where $\lambda$ is in Å and $B$ in Gauss, $g$ is the Landé factor.

$B$-field sensitivity $\propto \lambda^2$, which is why IR lines are much more sensitive.
B-필드 감도가 $\lambda^2$에 비례하므로, 근적외선이 가시광보다 훨씬 민감하다.

In [ ]:
def zeeman_splitting(wavelength_A, g, B_gauss):
    """Compute Zeeman wavelength shift in Ångströms.

    Args:
        wavelength_A: Rest wavelength in Ångström.
        g: Landé g-factor (dimensionless).
        B_gauss: Magnetic field strength in Gauss.

    Returns:
        Wavelength shift Δλ_B in Ångström.
    """
    return 4.67e-13 * g * wavelength_A ** 2 * B_gauss

wavelengths = [5000, 6173, 8700, 10500, 12300, 15650, 22000]
labels = ['Fe I 5000 (opt)', 'Fe I 6173 (opt)', 'Ca II 8700 (NIR)',
          'Fe I 10500 (NIR)', 'Fe I 12300 (NIR)', 'Fe I 15650 (NIR)', 'Ti I 22000 (IR)']

B_range = np.logspace(1, 4, 100)
plt.figure()
for lam, lab in zip(wavelengths, labels):
    dl = zeeman_splitting(lam, g=2.5, B_gauss=B_range)
    dv = C_KMS * dl / lam
    plt.loglog(B_range, dv, label=f'{lab}')
plt.axhline(1.0, color='k', linestyle=':', alpha=0.5, label='1 km/s (detectable)')
plt.xlabel('B (Gauss)')
plt.ylabel('Δv_B = c·Δλ/λ  (km/s)')
plt.title('Zeeman velocity shift vs B-field (g=2.5) / Zeeman 속도 편이 vs 자기장')
plt.legend(fontsize=9)
plt.grid(alpha=0.3, which='both')
plt.show()

print('At B = 1000 G, g = 2.5:')
for lam, lab in zip(wavelengths, labels):
    dl = zeeman_splitting(lam, 2.5, 1000)
    dv = C_KMS * dl / lam
    print(f'  {lab:30s}: Δλ = {dl*1000:7.2f} mÅ, Δv = {dv:5.2f} km/s')

## 2. Stokes I and V Profile Synthesis / Stokes I, V 프로파일 합성

Gaussian absorption line with Zeeman-split σ components (±Δλ_B) and central π component.

Weak-field approximation for Stokes V:
$$
V(\lambda) \approx -g_{\rm eff} \Delta\lambda_B \cos\theta \, \frac{dI}{d\lambda}
$$
where $\theta$ is the angle between $B$ and the line of sight.

Stokes V는 I의 미분에 비례 — 이 때문에 약한 필드에서도 검출 가능.
Stokes V is proportional to the derivative of I — which is why even weak fields are detectable.

In [ ]:
def gaussian_line(wavelength_grid, lam0, depth, width):
    """Gaussian absorption line profile.

    Args:
        wavelength_grid: Wavelength grid (same units as lam0).
        lam0: Rest wavelength.
        depth: Line depth (0 < depth < 1).
        width: Gaussian width σ.

    Returns:
        Intensity I(λ), normalized to 1 at continuum.
    """
    return 1.0 - depth * np.exp(-0.5 * ((wavelength_grid - lam0) / width) ** 2)

def zeeman_triplet_I(wavelength_grid, lam0, depth, width, dlam_B):
    """Stokes I for a simple Zeeman triplet (π + 2 σ components)."""
    pi_comp = 0.5 * depth * np.exp(-0.5 * ((wavelength_grid - lam0) / width) ** 2)
    sig_p = 0.25 * depth * np.exp(-0.5 * ((wavelength_grid - lam0 - dlam_B) / width) ** 2)
    sig_m = 0.25 * depth * np.exp(-0.5 * ((wavelength_grid - lam0 + dlam_B) / width) ** 2)
    return 1.0 - (pi_comp + sig_p + sig_m)

def stokes_V_weak(wavelength_grid, I_profile, g, dlam_B, cos_theta=1.0):
    """Weak-field Stokes V = -g·Δλ_B·cos(θ)·dI/dλ."""
    dIdl = np.gradient(I_profile, wavelength_grid)
    return -g * dlam_B * cos_theta * dIdl

lam0 = 6173.0
width = 0.08
depth = 0.6
g_eff = 2.5
B_values = [0, 500, 2000, 5000]
grid = np.linspace(lam0 - 1.0, lam0 + 1.0, 600)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 9), sharex=True)
for B in B_values:
    dlam_B = zeeman_splitting(lam0, g_eff, B)
    I = zeeman_triplet_I(grid, lam0, depth, width, dlam_B)
    V = stokes_V_weak(grid, I, g_eff, dlam_B, cos_theta=1.0)
    ax1.plot(grid, I, label=f'B = {B} G')
    ax2.plot(grid, V, label=f'B = {B} G')
ax1.set_ylabel('Stokes I / continuum')
ax1.set_title(f'Zeeman-split Fe I {lam0:.0f} Å (g={g_eff}) / Zeeman 분리된 Fe I 선')
ax1.legend()
ax1.grid(alpha=0.3)
ax2.set_xlabel('Wavelength (Å)')
ax2.set_ylabel('Stokes V (weak-field)')
ax2.axhline(0, color='k', lw=0.5)
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Optical vs Near-IR Sensitivity / 광학 vs 근적외선 민감도

같은 B=1 kG에서 가시광(5000 Å)과 근적외선(12000 Å) 프로파일을 비교.
Compare profiles at B=1 kG in optical (5000 Å) and near-IR (12000 Å).

IR에서는 Zeeman 분리가 크고 기준 thermal 확대보다 훨씬 크기 때문에 σ 성분이 분리되어 관측된다.
In IR, Zeeman splitting exceeds thermal broadening so the σ components resolve into distinct peaks.

In [ ]:
B_test = 1000.0
g = 2.5
T_dopp_kms = 3.0

lam_opt = 5000.0
lam_ir = 12000.0
sigma_opt = T_dopp_kms / C_KMS * lam_opt
sigma_ir = T_dopp_kms / C_KMS * lam_ir
dlam_opt = zeeman_splitting(lam_opt, g, B_test)
dlam_ir = zeeman_splitting(lam_ir, g, B_test)

print(f'At B = {B_test} G, thermal Doppler σ = {T_dopp_kms} km/s:')
print(f'  Optical  {lam_opt:.0f} Å: σ_thermal = {sigma_opt*1000:.1f} mÅ, Δλ_B = {dlam_opt*1000:.1f} mÅ → ratio {dlam_opt/sigma_opt:.2f}')
print(f'  Near-IR  {lam_ir:.0f} Å: σ_thermal = {sigma_ir*1000:.1f} mÅ, Δλ_B = {dlam_ir*1000:.1f} mÅ → ratio {dlam_ir/sigma_ir:.2f}')

grid_opt = np.linspace(lam_opt - 0.8, lam_opt + 0.8, 600)
grid_ir = np.linspace(lam_ir - 2.0, lam_ir + 2.0, 600)
I_opt = zeeman_triplet_I(grid_opt, lam_opt, 0.6, sigma_opt, dlam_opt)
I_ir = zeeman_triplet_I(grid_ir, lam_ir, 0.6, sigma_ir, dlam_ir)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(grid_opt - lam_opt, I_opt, 'b-')
ax1.set_title(f'Optical {lam_opt:.0f} Å, B={B_test} G / 가시광')
ax1.set_xlabel('Δλ (Å)')
ax1.set_ylabel('Stokes I')
ax1.grid(alpha=0.3)

ax2.plot(grid_ir - lam_ir, I_ir, 'r-')
ax2.set_title(f'Near-IR {lam_ir:.0f} Å, B={B_test} G / 근적외선')
ax2.set_xlabel('Δλ (Å)')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Zeeman vs Rotational Broadening / Zeeman vs 회전 확대

Zeeman 검출 임계는 Doppler(열+회전) 확대보다 Zeeman 분리가 커질 때:
Zeeman detection threshold: when Zeeman splitting exceeds thermal + rotational Doppler broadening:

$$
\Delta\lambda_B \gtrsim \sigma_{\rm Doppler} = \frac{\lambda}{c}\sqrt{v_T^2 + (v\sin i)^2}
$$

At what B does this happen for a given $v\sin i$? Critical B.

주어진 $v\sin i$에 대해 B_critical을 계산.

In [ ]:
def B_critical(wavelength_A, g, vsini_kms, v_thermal_kms=3.0):
    """Minimum detectable B where Zeeman shift ~ Doppler broadening."""
    v_dopp = np.sqrt(v_thermal_kms ** 2 + vsini_kms ** 2)
    sigma_lam = wavelength_A * v_dopp / C_KMS
    return sigma_lam / (4.67e-13 * g * wavelength_A ** 2)

vsini_range = np.logspace(0, 2, 100)
plt.figure()
for lam, label in [(5000, 'Optical 5000 Å'),
                   (8700, 'Ca II 8700 Å'),
                   (12300, 'FeH 12300 Å'),
                   (22000, 'Ti I 22000 Å')]:
    Bc = B_critical(lam, 2.5, vsini_range)
    plt.loglog(vsini_range, Bc, label=label)
plt.xlabel('v sin i (km/s)')
plt.ylabel('B_critical (Gauss)')
plt.title('Minimum detectable B vs rotational broadening / 회전 확대에 따른 검출 임계')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

print('Critical B (g=2.5) for solar vsini ≈ 2 km/s:')
for lam in [5000, 8700, 12300, 22000]:
    Bc = B_critical(lam, 2.5, 2.0)
    print(f'  λ = {lam:5d} Å: B_crit ≈ {Bc:7.0f} G')

## 5. Rossby Number and Activity / Rossby 수와 활동도

회전-활동도 관계는 Rossby 수 $R_o = P_{\rm rot} / \tau_{\rm conv}$로 정리된다:
The rotation-activity relation is organized by the Rossby number:

- 빠른 회전 ($R_o \lesssim 0.1$): 포화 영역, $L_X/L_{bol} \approx 10^{-3}$
- 느린 회전 ($R_o \gtrsim 0.1$): 비포화, $L_X/L_{bol} \propto R_o^{-2}$

Fast rotators saturate at $L_X/L_{bol} \sim 10^{-3}$; slow rotators follow $R_o^{-2}$.

In [ ]:
def activity_vs_rossby(Ro, Ro_sat=0.13, sat_level=-3.0, slope=-2.0):
    """Piecewise log L_X/L_bol vs Rossby number."""
    log_act = np.where(Ro < Ro_sat,
                       sat_level,
                       sat_level + slope * np.log10(Ro / Ro_sat))
    return log_act

Ro_grid = np.logspace(-2.5, 0.5, 200)
log_LX = activity_vs_rossby(Ro_grid)

plt.figure()
plt.semilogx(Ro_grid, log_LX, 'b-', lw=2)
plt.axvline(0.13, color='r', linestyle='--', alpha=0.6, label='$R_o^{sat} \\approx 0.13$')
plt.axvline(2.0, color='orange', linestyle=':', label='Sun ($R_o \\approx 2$)')
plt.scatter([2.0], [activity_vs_rossby(2.0)], marker='*', s=200, color='gold', edgecolor='k', label='Sun', zorder=5)
plt.xlabel('Rossby number $R_o = P_{rot} / \\tau_{conv}$')
plt.ylabel('$\\log(L_X / L_{bol})$')
plt.title('Rotation-activity relation / 회전-활동도 관계')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

print(f'Sun (R_o ≈ 2.0):     log(L_X/L_bol) ≈ {activity_vs_rossby(2.0):.2f}')
print(f'Saturated (R_o=0.05): log(L_X/L_bol) ≈ {activity_vs_rossby(0.05):.2f}')

## 6. Zeeman Doppler Imaging (ZDI) Concept / ZDI 개념

항성 표면에 자기 반점이 있으면, 별이 회전하면서 반점이 앞->중심->뒤로 이동하고 해당 Stokes V 신호가 도플러 편이되어 시선 속도축을 따라 이동.

With a magnetic spot on the stellar surface, as the star rotates the spot moves front → center → back and its Stokes V signal drifts in Doppler velocity — this is the basis of ZDI.

시간분해 Stokes V 스펙트럼(time-resolved Stokes V)은 별 표면 자기장 지도를 역산하는 데 사용된다.

In [ ]:
def stellar_spot_Stokes_V(phase, v_rot_kms, spot_lat_deg, spot_lon_deg,
                          B_spot_G, lam0, g, velocity_grid_kms):
    """Simple forward model of Stokes V from a single magnetic spot.

    Args:
        phase: Rotational phase (0..1).
        v_rot_kms: v sin i.
        spot_lat_deg, spot_lon_deg: Spot location.
        B_spot_G: Spot magnetic field.
        lam0: Rest wavelength (Å).
        g: Landé factor.
        velocity_grid_kms: Velocity grid (km/s) for the Stokes V profile.

    Returns:
        Stokes V(v) at the given phase.
    """
    lon_rot = np.deg2rad(spot_lon_deg - 360 * phase)
    lat = np.deg2rad(spot_lat_deg)
    v_proj = v_rot_kms * np.cos(lat) * np.sin(lon_rot)
    visibility = np.maximum(0, np.cos(lat) * np.cos(lon_rot))
    B_los = B_spot_G * np.cos(lat) * np.cos(lon_rot)
    dlam_B = zeeman_splitting(lam0, g, B_los)
    dv_B = C_KMS * dlam_B / lam0
    sigma_v = 3.0
    V_profile = -visibility * dv_B * (velocity_grid_kms - v_proj) / sigma_v ** 2 * \
                np.exp(-0.5 * ((velocity_grid_kms - v_proj) / sigma_v) ** 2)
    return V_profile

v_grid = np.linspace(-40, 40, 400)
phases = np.linspace(0, 1, 9)

plt.figure(figsize=(9, 10))
offset = 0
for p in phases:
    V = stellar_spot_Stokes_V(p, v_rot_kms=30.0, spot_lat_deg=45,
                              spot_lon_deg=0, B_spot_G=2000,
                              lam0=6173.0, g=2.5, velocity_grid_kms=v_grid)
    plt.plot(v_grid, V + offset, label=f'φ = {p:.2f}')
    offset += 5e-4
plt.xlabel('Velocity (km/s)')
plt.ylabel('Stokes V (offset by phase)')
plt.title('Time-resolved Stokes V — ZDI input / 시간분해 Stokes V — ZDI 입력')
plt.legend(loc='upper right', fontsize=8)
plt.grid(alpha=0.3)
plt.show()

## Summary / 요약

1. **Zeeman splitting** — B=1 kG, g=2.5에서 가시광(5000 Å) ≈12 mÅ vs 근적외선(12000 Å) ≈67 mÅ. IR의 B-감도가 ~5배.
2. **Stokes V weak-field** — I의 미분에 비례, 약한 필드에서도 광도 1/1000 수준까지 검출.
3. **Zeeman vs rotational** — 빠른 회전자(M dwarf 30 km/s)에서는 Zeeman 검출 임계가 ~1-2 kG 수준.
4. **Rossby-activity** — $R_o \lesssim 0.13$ 포화, 이후 $R_o^{-2}$. 태양은 $R_o \sim 2$에서 $\log(L_X/L_{bol}) \approx -7$.
5. **ZDI** — 회전 단계별 Stokes V 스펙트럼이 표면 자기장 지도의 역산 입력.

### References
- Reiners, A. (2012). *Observations of Cool-Star Magnetic Fields*. LRSP 9, 1. https://doi.org/10.12942/lrsp-2012-1
- Landstreet, J. D. (1992). *Magnetic fields at the surfaces of stars*. A&A Rev., 4, 35.
- Noyes, R. W. et al. (1984). *Rotation, convection, and magnetic activity in lower main-sequence stars*. ApJ, 279, 763.
- Pizzolato, N. et al. (2003). *The stellar activity-rotation relationship revisited*. A&A, 397, 147.